<a href="https://colab.research.google.com/github/chisangachileshe623-ship-it/Africa-Digital-Divide/blob/main/01_africa_digital_divide.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print ("Africa's Digital Divide")
print ("Project 1")

Africa's Digital Divide
Project 1


In [4]:
import requests

url = "https://api.worldbank.org/v2/country/all/indicator/IT.NET.USER.ZS?format=json&per_page=20000"

response = requests.get(url)

print("Status code:", response.status_code)
print("Response type:", response.headers.get("content-type"))
print(response.text[:500])

Status code: 200
Response type: application/json;charset=utf-8
[{"page":1,"pages":1,"per_page":20000,"total":17490,"sourceid":"2","lastupdated":"2026-07-13"},[{"indicator":{"id":"IT.NET.USER.ZS","value":"Individuals using the Internet (% of population)"},"country":{"id":"ZH","value":"Africa Eastern and Southern"},"countryiso3code":"AFE","date":"2025","value":30.4,"unit":"","obs_status":"","decimal":0},{"indicator":{"id":"IT.NET.USER.ZS","value":"Individuals using the Internet (% of population)"},"country":{"id":"ZH","value":"Africa Eastern and Southern"},"c


In [21]:
import requests
import pandas as pd

indicators = {
    "IT.NET.USER.ZS": "internet_users",
    "NY.GDP.PCAP.KD": "gdp_per_capita",
    "EG.ELC.ACCS.ZS": "electricity_access",
    "SP.URB.TOTL.IN.ZS": "urban_population"
}

dataframes = []

for indicator, name in indicators.items():

    url = f"https://api.worldbank.org/v2/country/all/indicator/{indicator}?format=json&per_page=20000"

    response = requests.get(url)

    response.raise_for_status()

    result = response.json()

    observations = result[1]

    df = pd.DataFrame(observations)

    df['country'] = df['country'].apply(lambda x: x['value'] if isinstance(x, dict) else x)

    df = df[[
        "countryiso3code",
        "country",
        "date",
        "value"
    ]]

    df = df.rename(columns={
        "value": name
    })

    dataframes.append(df)


data = dataframes[0]

for df in dataframes[1:]:
    data = data.merge(
        df,
        on=["countryiso3code", "country", "date"],
        how="outer"
    )

data = data.rename(columns={"date": "year"})

data["year"] = pd.to_numeric(data["year"])

data = data[
    (data["year"] >= 2000) &
    (data["year"] <= 2024)
]

print("Dataset shape:", data.shape)

data.head()

Dataset shape: (6625, 7)


,countryiso3code,country,year,internet_users,gdp_per_capita,electricity_access,urban_population
40,,High income,2000,NaN,30253.840917,99.441643,76.124359
41,,High income,2001,NaN,30574.410855,99.466443,76.383580
42,,High income,2002,NaN,30920.087105,99.486329,76.705729
43,,High income,2003,NaN,31459.581730,99.528425,77.060638
44,,High income,2004,NaN,32386.375080,99.523189,77.409597


In [6]:
data.shape

(6625, 7)

In [7]:
data.columns

Index(['countryiso3code', 'country', 'year', 'internet_users',
       'gdp_per_capita', 'electricity_access', 'urban_population'],
      dtype='object')

In [8]:
data["country"].nunique()

265

In [9]:
import pandas as pd

# African countries (ISO3 country codes)

africa_codes = [
    "DZA", "AGO", "BEN", "BWA", "BFA", "BDI", "CPV", "CMR",
    "CAF", "TCD", "COM", "COG", "COD", "CIV", "DJI", "EGY",
    "GNQ", "ERI", "SWZ", "ETH", "GAB", "GMB", "GHA", "GIN",
    "GNB", "KEN", "LSO", "LBR", "LBY", "MDG", "MWI", "MLI",
    "MRT", "MUS", "MAR", "MOZ", "NAM", "NER", "NGA", "RWA",
    "STP", "SEN", "SYC", "SLE", "SOM", "ZAF", "SSD", "SDN",
    "TZA", "TGO", "TUN", "UGA", "ZMB", "ZWE"
]

africa_data = data[data["countryiso3code"].isin(africa_codes)].copy()

print("Rows:", africa_data.shape[0])
print("Countries:", africa_data["country"].nunique())
print("Years:", africa_data["year"].min(), "-", africa_data["year"].max())

Rows: 1350
Countries: 54
Years: 2000 - 2024


In [10]:
# checking the African countries in the dataset

print ("Number of countries:", africa_data["country"].nunique())
print ("\nCountries:")
print(sorted(africa_data["country"].unique()))


Number of countries: 54

Countries:
['Algeria', 'Angola', 'Benin', 'Botswana', 'Burkina Faso', 'Burundi', 'Cabo Verde', 'Cameroon', 'Central African Republic', 'Chad', 'Comoros', 'Congo, Dem. Rep.', 'Congo, Rep.', "Cote d'Ivoire", 'Djibouti', 'Egypt, Arab Rep.', 'Equatorial Guinea', 'Eritrea', 'Eswatini', 'Ethiopia', 'Gabon', 'Gambia, The', 'Ghana', 'Guinea', 'Guinea-Bissau', 'Kenya', 'Lesotho', 'Liberia', 'Libya', 'Madagascar', 'Malawi', 'Mali', 'Mauritania', 'Mauritius', 'Morocco', 'Mozambique', 'Namibia', 'Niger', 'Nigeria', 'Rwanda', 'Sao Tome and Principe', 'Senegal', 'Seychelles', 'Sierra Leone', 'Somalia, Fed. Rep.', 'South Africa', 'South Sudan', 'Sudan', 'Tanzania', 'Togo', 'Tunisia', 'Uganda', 'Zambia', 'Zimbabwe']


In [11]:
# checking missing values for each variable

print("Missing values:")
print(africa_data.isnull().sum())


Missing values:
countryiso3code        0
country                0
year                   0
internet_users        53
gdp_per_capita        43
electricity_access    16
urban_population       0
dtype: int64


In [12]:
# calculating the percentage of missing values

missing_summary = pd.DataFrame({
    "Missing Values": africa_data.isnull().sum(),
    "Percentage": (africa_data.isnull().mean() * 100).round(2)
})

missing_summary

,Missing Values,Percentage
countryiso3code,0,0.00
country,0,0.00
year,0,0.00
internet_users,53,3.93
gdp_per_capita,43,3.19
electricity_access,16,1.19
urban_population,0,0.00


In [13]:
# Checking how many complete observations are there

print("Total observations:", len(africa_data))
print("Complete observations:", africa_data.dropna().shape[0])
print("Observations with at least one missing value:", africa_data.isnull().any(axis=1).sum())

Total observations: 1350
Complete observations: 1262
Observations with at least one missing value: 88


In [14]:
missing_values = africa_data.isnull().sum()

print("Missing values by variables:")
print(missing_values)

Missing values by variables:
countryiso3code        0
country                0
year                   0
internet_users        53
gdp_per_capita        43
electricity_access    16
urban_population       0
dtype: int64


In [15]:
# Checking whether the dataset variable exists
print("africa_data" in globals())

True


In [16]:
print("africa_data" in globals())

True


In [17]:
# Checking how many missing values each column has
df.isnull().sum()

,0
countryiso3code,0
country,0
date,0
urban_population,66


In [18]:
%whos

Variable          Type         Data/Info
----------------------------------------
africa_codes      list         n=54
africa_data       DataFrame          countryiso3code   c<...>\n[1350 rows x 7 columns]
data              DataFrame          countryiso3code    <...>\n[6625 rows x 7 columns]
dataframes        list         n=4
df                DataFrame          countryiso3code    <...>n[17490 rows x 4 columns]
indicator         str          SP.URB.TOTL.IN.ZS
indicators        dict         n=4
missing_summary   DataFrame                        Missi<...>            0        0.00
missing_values    Series       countryiso3code        0\<...>ion       0\ndtype: int64
name              str          urban_population
np                module       <module 'numpy' from '/us<...>kages/numpy/__init__.py'>
observations      list         n=17490
pd                module       <module 'pandas' from '/u<...>ages/pandas/__init__.py'>
plt               module       <module 'matplotlib.pyplo<...>es/mat

In [23]:
df[df["urban_population"].isnull()]

,countryiso3code,country,date,urban_population
2244,,Not classified,2025,NaN
2245,,Not classified,2024,NaN
2246,,Not classified,2023,NaN
2247,,Not classified,2022,NaN
2248,,Not classified,2021,NaN
...,...,...,...,...
2305,,Not classified,1964,NaN
2306,,Not classified,1963,NaN
2307,,Not classified,1962,NaN
2308,,Not classified,1961,NaN


In [24]:
df["country"].value_counts().tail()

,count
country,
European Union,66
Europe & Central Asia (IDA & IBRD countries),66
Europe & Central Asia (excluding high income),66
Europe & Central Asia,66
Euro area,66


In [25]:
df[df["country"] == "Not classified"].shape

(66, 4)

In [26]:
df[df["country"] == "Not classified"]["countryiso3code"].unique()

array([''], dtype=object)

In [27]:
df = df[df["country"] != "Not classified"].copy()

In [28]:
df.shape

(17424, 4)

In [29]:
df.isnull().sum()

,0
countryiso3code,0
country,0
date,0
urban_population,0
